# 02 Dijkstra：带权最短路（松弛、堆、有向边）


本课把 BFS 的 FIFO 换成 **按已付代价 g 排序的优先队列**。边可以有不同正长度，图可以有向。

---

## 1. 定义

**Dijkstra 算法**（Dijkstra, 1959）计算加权图上从 \(s\) 到各点（或到 \(t\)）的最短路。

要求：**边权 \(w(u,v)\ge 0\)**。有负权边时结论不成立，应改 Bellman–Ford。

记 \(g(v)\) 为目前已知的 \(s\leadsto v\) 代价上界。开始 \(g(s)=0\)，其余 \(+\infty\)。反复：

1. 在「尚未关闭」的点里选 \(g\) 最小的 \(u\) **关闭**（出堆）。
2. 对每条出边 \(u\to v\) 做 **松弛**：若 \(g(u)+w(u,v) < g(v)\)，则更新 \(g(v)\) 并把 \(v\) 的前驱设为 \(u\)。

**定理（非负权）：** \(u\) 被关闭时，\(g(u)\) 等于真正最短路长 \(\delta(s,u)\)。

直觉：若还存在更短的 \(s\leadsto u\)，这条路上第一个未关闭点 \(x\) 会有 \(g(x)<g(u)\)，堆会先弹出 \(x\) 而不是 \(u\)。非负权保证路上代价不会越走越便宜到「后面的点比前面还小」。

---

## 2. BFS 错在哪（数值例子）

```
s --1--> a --1--> t     总长 2，两步
s --10--> t             总长 10，一步
```

BFS 按步数先看见 `s→t`，宣布最短，错。Dijkstra 先关闭 s，松弛得 \(g(a)=1, g(t)=10\)；再关闭 a（g=1 更小），松弛 \(g(t)=\min(10,1+1)=2\)。正确。

等权时 Dijkstra 与 BFS **选出的代价相同**（实现仍用堆，常数更大）。

---

## 3. 优先队列：两种实现

教科书常写 decrease-key：v 的 g 变小就改堆里那个元素。Python `heapq` **没有** decrease-key。

教学与工程常用 **惰性堆**：

- 每次 g 变小就 `heappush` 一个新 `(g, v)`，旧的更大 g 留在堆里。
- 弹出时若 `v` 已经关闭（或弹出的 g 不是当前 `g_score[v]`），跳过。

正确性不变，最坏堆里 \(O(E)\) 个条目，时间 \(O(E\log E)\)，稀疏图可接受。

元组写成 `(g, counter, node)`：`counter` 是单调整数，避免 `g` 相同去比较 `node`（点可能是坐标元组，能比；若是不可比对象会崩）。

---

## 4. 伪代码（惰性堆，与下面 Python 一致）

```
function Dijkstra(G, s, t):
    g[s] ← 0; 其余视为 ∞
    came_from[s] ← NIL
    push heap (0, s)
    closed ← ∅
    while heap 非空:
        (gu, u) ← pop 最小 g
        if u ∈ closed: continue
        closed.add(u)
        if u = t: return Reconstruct(came_from, t)
        for each 边 (u, v, w):
            alt ← gu + w
            if alt < g[v]:          # 松弛成功
                g[v] ← alt
                came_from[v] ← u
                push heap (alt, v)
    return FAILURE
```

有向图：循环只扫 **出边**。没有反向边就不会走回去。把无向边存成两条有向边。

复杂度：二叉堆 \(O((V+E)\log V)\)（经典 decrease-key 版）或惰性堆 \(O(E\log E)\)。

---

## 5. 手算（有向）

课程图 `topo_oneway`：环 I0→I1→I2→I3→I4→I5→I0，另有 I1→I4，I4↔G0，I2↔G1。边权 1。求 I0→G1。

一种最短路：I0–I1–I2–G1，g=3。I1 不能一步到 I0。请先打印邻居再手填 g。


In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "lib").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))


In [ ]:
from lib.maps import topo_oneway

g = topo_oneway()
print("出边列表（有向）：")
for name in sorted(g.nodes):
    print(f"  {name}: {g.neighbors(name)}")


## 6. 逐行实现

松弛那四行是算法心脏：`tentative = g_cur + cost` 就是 \(g(u)+w(u,v)\)；只有严格更小才改前驱。相等时不改，保持先发现的父亲（稳定）。


In [ ]:
import heapq


def reconstruct(came_from, goal):
    node = goal
    path = [node]
    while came_from[node] is not None:
        node = came_from[node]
        path.append(node)
    path.reverse()
    return path


def dijkstra(graph, start, goal, verbose=True):
    heap = []
    counter = 0
    g_score = {start: 0.0}
    came_from = {start: None}
    heapq.heappush(heap, (0.0, counter, start))
    closed = set()
    step = 0
    while heap:
        g_cur, _, current = heapq.heappop(heap)
        if current in closed:
            # 惰性堆：过期条目。更小的 g 已经处理过 current。
            continue
        closed.add(current)
        step += 1
        if verbose:
            print(f"关闭 {current:3}  g={g_cur:.1f}  closed={sorted(closed, key=str)}")
        if current == goal:
            path = reconstruct(came_from, goal)
            if verbose:
                print("路径", path, "代价", g_cur)
            return path, g_cur
        for nxt, cost in graph.neighbors(current):
            # graph.neighbors 只给合法出边：有向、不碰墙都在图 API 里处理完了
            tentative = g_cur + cost
            if tentative < g_score.get(nxt, float("inf")):
                g_score[nxt] = tentative
                came_from[nxt] = current
                counter += 1
                heapq.heappush(heap, (tentative, counter, nxt))
                if verbose:
                    print(f"    松弛 {current} -{cost}-> {nxt}  新g={tentative:.1f}")
    if verbose:
        print("不可达")
    return None, float("inf")


path, cost = dijkstra(g, "I0", "G1")


关闭顺序应接近：I0(g=0) → I1(1) → I2 与 I4(2) → G1(3)… 具体并列谁先由 counter 决定。

**负权为何失败：** 关闭 u 之后，若存在 u 经一条负边绕到更便宜，定理的「路上 g 递增」破了，已关闭点的 g 还可能再降，算法不会回去改。

---

## 7. 与 BFS、库函数对照


In [ ]:
from lib.maps import grid_open
from lib.search import bfs, dijkstra as dijkstra_lib

grid = grid_open(8, 8)
b = bfs(grid, (0, 0), (7, 7), record=False)
d = dijkstra_lib(grid, (0, 0), (7, 7), record=False)
print("等权网格 BFS 边数", b.cost, "Dijkstra", d.cost, "相等?", b.cost == d.cost)
print("I1 的出边有没有 I0（应为没有）:", [n for n, _ in g.neighbors("I1")])


In [ ]:
from lib.studio.widget import PlannerStudio
from lib.search import dijkstra as dijkstra_lib
from lib.maps import grid_wall

gw = grid_wall()
r = dijkstra_lib(gw, (0, 0), (7, 7))
print("墙图代价", r.cost, "扩展", r.expanded)
studio = PlannerStudio()
studio.show_search({**gw.to_studio(), "start": [0, 0], "goal": [7, 7]}, r, "Dijkstra")
studio


库 `lib/search.py` 的 `best_first(..., heuristic=lambda a,b:0)` 就是 Dijkstra：按 `g + 0` 出队。下一课把 0 换成 h。

## 条目小结

| 项目 | 内容 |
|------|------|
| 前提 | \(w\ge 0\) |
| 操作 | 关最小 g 点 + 松弛出边 |
| 堆 | 惰性插入，过期弹出跳过 |
| 有向 | 只迭代出边 |
| 下一课 | \(f=g+h\) → A* |

## 练习

1. 纸上跑 `s-1-a-1-t` 与 `s-10-t`，写出每次关闭和松弛。
2. 若 `tentative == g_score[nxt]` 也更新 came_from，路径对不对？唯一吗？
3. 从 G1 搜回 I0，必须绕环，不能走 I2←I1。打印路径验证。
